In [ ]:
# %%
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from collections import Counter

import matplotlib.pyplot as plt

# Set up the logging so that we can see progress
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)


In [ ]:
def viz_by_group(viz_args, **kwargs):
    ind_flood = kwargs['ind_flood']
    start_str, end_str = kwargs['in_flood_start_end'][ind_flood]
    n_days = (pd.to_datetime(end_str) - pd.to_datetime(start_str)).days

    n_clusters = kwargs['n_clusters']
    save_root = kwargs['save_root']

    # yymmdd_list = viz_args['yymmdd_list']
    yymmdd_list = np.array(viz_args['yymmdd_list']) - 30
    yymmdd_list_str = viz_args['yymmdd_list_str']
    myarray = viz_args['myarray']
    labels = viz_args['labels']
    labels_N = viz_args['labels_N']
    meancenter4 = viz_args['meancenter4']
    std4 = viz_args['std4']

    def plot_result_group(g, color_g):
        # colors = ['#225bb2', '#60B5FF', '#FF8282', '#BB3E00']
        colors = ['#66a7a6', '#e16503']

        plt.axvspan(0, n_days, color='lightgrey', alpha=0.2, zorder=0)

        for i in range(len(myarray)):
            for n in range(n_clusters):
                if labels[i]==g:
                    plt.plot(yymmdd_list, myarray[i], color=colors[color_g], alpha=0.4, linewidth=0.05)

        for i in range(len(labels_N)):
            if i == g:
                plt.plot(yymmdd_list, meancenter4[i], marker='o', markersize=3, color=colors[color_g], linewidth=1.5,
                        label = "%s neighborhoods"%(Counter(labels)[i]))

                # fill_between
                x1 = np.array(np.arange(len(std4[i]))) - 30
                f1 = meancenter4[i]+std4[i]
                f2 = meancenter4[i]-std4[i]
                plt.fill_between(x1, f1, f2, alpha=0.2, color=colors[color_g])
                plt.fill_between(x1, meancenter4[i], alpha=0.1, color='k')

        plt.plot(yymmdd_list, [0]*myarray.shape[1], color='Black', linewidth = 1, linestyle='--', alpha=0.6)
        plt.legend(loc='upper left', fontsize=14)
        plt.tick_params(axis='both', which='major', labelsize=14)
        y_max = max(myarray.max(), abs(myarray.min())) + 0.1
        plt.ylim(-y_max,y_max)
        plt.grid(axis='y',linewidth=0.5, alpha=0.5)
    
    sup = kwargs['sup']
    fig = plt.figure(figsize=(15,6), dpi=300)
    plt.suptitle(sup, fontsize=16)

    # Cluster label integers are not ordered consistently across events (the
    # decrease cluster is label 1 for one event and label 0 for another), so
    # derive which label is which from the sign of each cluster's mean trend
    # rather than hardcoding label indices per event. Subplot 1 / colors[0]
    # (teal) is always the decrease cluster; subplot 2 / colors[1] (orange)
    # is always the increase cluster. This derivation depends on two
    # preconditions: exactly two clusters (only two panels/colors exist), and
    # the two cluster means must be distinguishable (not tied, not NaN) so
    # that argmin/argmax do not collapse onto the same label.
    if n_clusters != 2:
        raise ValueError(
            "This figure draws exactly two panels (decrease, increase); "
            f"got n_clusters={n_clusters}."
        )
    cluster_mean = [myarray[labels == n].mean() for n in range(n_clusters)]
    decrease_label = int(np.argmin(cluster_mean))
    increase_label = int(np.argmax(cluster_mean))
    if decrease_label == increase_label:
        raise ValueError(
            "Cannot tell the decrease cluster from the increase cluster; "
            f"cluster means are {cluster_mean}."
        )

    plt.subplot(1,2,1)
    plot_result_group(decrease_label, color_g=0)

    plt.subplot(1,2,2)
    plot_result_group(increase_label, color_g=1)

    plt.savefig(os.path.join(save_root, 'by_group.png'), dpi=300, bbox_inches='tight')
    plt.show()
    # plt.close()

In [ ]:
def viz_pipeline(re, **kwargs):
    n_clusters = kwargs['n_clusters']

    myarray = re.iloc[:, 1:-1].to_numpy()
    labels = re['label'].to_numpy()

    labels_N = np.zeros([n_clusters,myarray.shape[0]])
    for n in range(n_clusters):
        for i in range(len(myarray)):
            if labels[i]==n:
                labels_N[n][i]=1
            else:
                labels_N[n][i]=0

    meancenter4 = np.zeros([n_clusters,myarray.shape[1]])
    std4 = np.zeros([n_clusters,myarray.shape[1]])

    for n in range(len(labels_N)):
        for i in range(len(myarray[0])):
            A=[]
            for j in range(len(myarray)):
                a=myarray[j][i]*labels_N[n][j]
                A.append(a)
            meancenter4[n][i] = sum(A)/sum(labels_N[n])
            std4[n][i] = np.std(np.trim_zeros(A))
    
    yymmdd_list = range(len(myarray[1]))
    yymmdd_list_str = re.columns[1:-1].tolist()

    viz_args = {
        'myarray': myarray,
        'labels': labels,
        'labels_N': labels_N,
        'meancenter4': meancenter4,
        'std4': std4,
        'yymmdd_list': yymmdd_list,
        'yymmdd_list_str': yymmdd_list_str,
    }

    # viz_all_groups(viz_args, **kwargs)
    viz_by_group(viz_args, **kwargs)
    # viz_all_trends(viz_args, **kwargs)

    return viz_args

In [ ]:
def process_pipeline(**kwargs):
    # suptitle
    ind_flood = kwargs['ind_flood']
    start_end = kwargs['in_flood_start_end'][ind_flood]
    sup = f'{ind_flood} ({start_end[0]} - {start_end[1]})'
    kwargs['sup'] = sup

    save_root = kwargs['save_root']
    clustering_res_path = os.path.join(save_root, 'clustering_result.csv')

    res = pd.read_csv(clustering_res_path)
    res = viz_pipeline(res, **kwargs)

    return res

In [ ]:
in_flood_start_end = {
    '4775': ('2019-07-01', '2019-07-05'),
    # '4859': ('2020-01-01', '2020-01-09'),
}
in_flood_no = [
    '4775',
    # '4859',
]

n_clusters = 2
data_root = '../outputs'

params = {
    'data_root': data_root,
    'n_clusters': n_clusters,
    'in_flood_start_end': in_flood_start_end,
}
for ind_flood in in_flood_no:
    logging.info(f'Processing {ind_flood}...')
    params['ind_flood'] = ind_flood
    ind_save_root = os.path.join(data_root, ind_flood, f'cluster_{n_clusters}')
    # os.makedirs(ind_save_root, exist_ok=True)
    params['save_root'] = ind_save_root

    res = process_pipeline(**params)

In [ ]:
in_flood_start_end = {
    # '4775': ('2019-07-01', '2019-07-05'),
    '4859': ('2020-01-01', '2020-01-09'),
}
in_flood_no = [
    # '4775',
    '4859',
]

n_clusters = 2
data_root = '../outputs'

params = {
    'data_root': data_root,
    'n_clusters': n_clusters,
    'in_flood_start_end': in_flood_start_end,
}
for ind_flood in in_flood_no:
    logging.info(f'Processing {ind_flood}...')
    params['ind_flood'] = ind_flood
    ind_save_root = os.path.join(data_root, ind_flood, f'cluster_{n_clusters}')
    # os.makedirs(ind_save_root, exist_ok=True)
    params['save_root'] = ind_save_root

    res = process_pipeline(**params)